## Functions

In [3]:
import os
import cv2
import json
import numpy as np
import math

_COLOR_STOPS = [
    (0.00, np.array([  0, 128,   0], dtype=float)),  # dark green
    (0.50, np.array([  0, 255, 255], dtype=float)),  # yellow
    (0.75, np.array([  0, 165, 255], dtype=float)),  # orange
    (1.00, np.array([  0,   0, 200], dtype=float)),  # dark red
]
def _lerp(a, b, t):
    return a + (b - a) * t

def color_for_value(v, vmin=0.0, vmax=12.0, gamma=1.0):
    """Map scalar v in [vmin,vmax] to BGR using green→yellow→orange→red."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        v = vmin
    t = 0.0 if vmax <= vmin else (float(v) - vmin) / (vmax - vmin)
    t = np.clip(t, 0.0, 1.0)
    if gamma != 1.0:
        t = t**gamma  # <1 emphasize lows, >1 emphasize highs

    # find surrounding stops
    for (t0, c0), (t1, c1) in zip(_COLOR_STOPS, _COLOR_STOPS[1:]):
        if t <= t1:
            local = 0.0 if t1 == t0 else (t - t0) / (t1 - t0)
            c = _lerp(c0, c1, local)
            return tuple(int(round(x)) for x in c)
    # fallback (shouldn't hit)
    return tuple(int(x) for x in _COLOR_STOPS[-1][1])

def visualize_hands_pose(frame, left_hand_instances, right_hand_instances, kpt_thresh=0.3):
    """
    Visualize hand poses (per-finger colors) and also return vector primitives.

    Returns:
        vis_frame   : frame with markings
        naked_frame : original frame (no markings)
        marks_only  : only the markings on black (same size as frame)
        primitives  : dict for vector export:
                      {
                        "frame_size": (W, H),
                        "items": [
                          {"type":"line","p1":(x1,y1),"p2":(x2,y2),"rgb":(r,g,b),"thick":px},
                          {"type":"circle","center":(x,y),"radius":px,"rgb":(r,g,b),"fill":(r,g,b)|None,"thick":px},
                          ...
                        ]
                      }
    """
    H, W = frame.shape[:2]
    naked_frame = frame.copy()
    overlay = np.zeros_like(frame)  # draw all markings here
    primitives = {"frame_size": (W, H), "items": []}

    # Define finger connections and colors (OpenCV BGR; primitives need RGB)
    finger_connections = {
        "thumb":  [(0, 1), (1, 2), (2, 3), (3, 4)],
        "index":  [(0, 5), (5, 6), (6, 7), (7, 8)],
        "middle": [(0, 9), (9, 10), (10, 11), (11, 12)],
        "ring":   [(0, 13), (13, 14), (14, 15), (15, 16)],
        "pinky":  [(0, 17), (17, 18), (18, 19), (19, 20)],
    }
    finger_colors_bgr = {
        "thumb":  (0, 0, 255),      # red
        "index":  (0, 255, 0),      # green
        "middle": (255, 0, 0),      # blue
        "ring":   (0, 255, 255),    # yellow
        "pinky":  (255, 0, 255),    # pink
    }
    wrist_bgr = (200, 200, 200)

    def bgr_to_rgb_triplet(bgr):
        b, g, r = bgr
        return (int(r), int(g), int(b))

    def _to_xy(a):
        a = np.asarray(a)
        if a.ndim == 3 and a.shape[0] == 1:
            a = a[0]
        return a

    def _to_1d(a):
        return np.asarray(a).squeeze()

    def _safe_pt(arr, idx):
        x, y = arr[idx]
        return (int(x), int(y))

    def draw_hand(instances):
        if instances is None:
            return

        keypoints = _to_xy(instances['keypoints'])
        scores    = _to_1d(instances['keypoint_scores'])
        n = min(len(keypoints), len(scores))
        if n == 0:
            return

        # Draw fingers
        for finger, connections in finger_connections.items():
            col_bgr = finger_colors_bgr[finger]
            col_rgb = bgr_to_rgb_triplet(col_bgr)

            for i, j in connections:
                if i < n and j < n and scores[i] > kpt_thresh and scores[j] > kpt_thresh:
                    p1 = _safe_pt(keypoints, i)
                    p2 = _safe_pt(keypoints, j)
                    # raster line
                    cv2.line(overlay, p1, p2, col_bgr, 2, lineType=cv2.LINE_AA)
                    # vector primitive
                    primitives["items"].append({
                        "type": "line",
                        "p1": (float(p1[0]), float(p1[1])),
                        "p2": (float(p2[0]), float(p2[1])),
                        "rgb": col_rgb,
                        "thick": 2.0,
                    })

            # joints (the distal joints for each connection list, wrist separately)
            joint_idxs = [c[1] for c in connections]
            for j in joint_idxs:
                if j < n and scores[j] > kpt_thresh:
                    x, y = _safe_pt(keypoints, j)
                    # raster circle
                    cv2.circle(overlay, (x, y), 3, col_bgr, -1, lineType=cv2.LINE_AA)
                    # vector primitive (filled)
                    primitives["items"].append({
                        "type": "circle",
                        "center": (float(x), float(y)),
                        "radius": 3.0,
                        "rgb": col_rgb,
                        "fill": col_rgb,
                        "thick": 0.0,
                    })

        # Wrist (0) in neutral gray
        if 0 < n and scores[0] > kpt_thresh:
            x, y = _safe_pt(keypoints, 0)
            cv2.circle(overlay, (x, y), 4, wrist_bgr, -1, lineType=cv2.LINE_AA)
            primitives["items"].append({
                "type": "circle",
                "center": (float(x), float(y)),
                "radius": 4.0,
                "rgb": bgr_to_rgb_triplet(wrist_bgr),
                "fill": bgr_to_rgb_triplet(wrist_bgr),
                "thick": 0.0,
            })

    # Draw both hands
    draw_hand(left_hand_instances)
    draw_hand(right_hand_instances)

    # Compose
    # before drawing:
    overlay = np.zeros_like(frame)                 # BGR colors
    mask    = np.zeros(frame.shape[:2], np.uint8)  # 8-bit mask

    # when you draw a line/circle, draw to BOTH overlay and mask:
    # example for a bone:
    cv2.line(overlay, pt1, pt2, color_bgr, bone_thickness, cv2.LINE_AA)
    cv2.line(mask,    pt1, pt2, 255,        bone_thickness, cv2.LINE_AA)

    # example for a filled joint:
    cv2.circle(overlay, (x,y), joint_radius, color_bgr, -1, cv2.LINE_AA)
    cv2.circle(mask,    (x,y), joint_radius, 255,       -1, cv2.LINE_AA)

    # example for an outline ring:
    if outline_thick > 0:
        cv2.circle(overlay, (x,y), joint_radius + outline_thick, (0,0,0), outline_thick, cv2.LINE_AA)
        cv2.circle(mask,    (x,y), joint_radius + outline_thick, 255,      outline_thick, cv2.LINE_AA)

    # final composite (no wash-out):
    vis_frame = naked_frame.copy()
    m = mask.astype(bool)
    vis_frame[m] = overlay[m]

    marks_only = overlay

    return vis_frame, naked_frame, marks_only, primitives


def process_video_from_json(video_path, hand_output_json_path, output_path, stride=1, preview=False):
    with open(hand_output_json_path, "r") as f:
        data = json.load(f)

    left_hand_instances = data["left_hand_instances"]
    right_hand_instances = data["right_hand_instances"]

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30  # fallback if fps==0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # ✅ make sure output folder exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # mp4 writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    if not video_writer.isOpened():
        raise RuntimeError(f"VideoWriter could not open file {output_path}")

    for frame_idx in range(0, total_frames, stride):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()
        if not success:
            print(f"⚠️ Could not read frame {frame_idx}, skipping.")
            continue

        print(f"Processing frame {frame_idx}/{total_frames}")

        # extract instances for this frame
        left_hand_instance = left_hand_instances[frame_idx]
        right_hand_instance = right_hand_instances[frame_idx]

        left_inst  = left_hand_instance.get("instances", [])
        left_inst  = left_inst[0] if len(left_inst) > 0 else None

        right_inst = right_hand_instance.get("instances", [])
        right_inst = right_inst[0] if len(right_inst) > 0 else None

        vis_frame = visualize_hands_pose(frame, left_inst, right_inst)

        # ensure size matches the writer
        if (vis_frame.shape[1], vis_frame.shape[0]) != (width, height):
            vis_frame = cv2.resize(vis_frame, (width, height))

        video_writer.write(vis_frame)

        if preview:
            # shrink for display, e.g. max width = 960px
            max_width = 960
            scale = max_width / vis_frame.shape[1]
            preview_frame = cv2.resize(
                vis_frame,
                (int(vis_frame.shape[1]*scale), int(vis_frame.shape[0]*scale))
            )

            cv2.imshow("Hand Pose", preview_frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    # cleanup
    cap.release()
    video_writer.release()
    if preview:
        cv2.destroyAllWindows()

    print(f"✅ Finished! Saved output video to: {output_path}")


def process_png_from_json(video_path, hand_output_json_path, output_path, frame_idx=0, preview=True):

    # ---- load detections ----
    with open(hand_output_json_path, "r") as f:
        data = json.load(f)

    left_hand_instances  = data.get("left_hand_instances", [])
    right_hand_instances = data.get("right_hand_instances", [])

    # ---- open video & sanity checks ----
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total_frames <= 0:
        raise ValueError(f"Video has no frames: {video_path}")

    if frame_idx < 0 or frame_idx >= total_frames:
        raise IndexError(f"frame_idx {frame_idx} out of range [0, {total_frames-1}]")

    # ---- seek & read frame ----
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        raise RuntimeError(f"Could not read frame {frame_idx} from {video_path}")

    # ---- pick instances for this frame (robust to missing/empty) ----
    def pick_instance(instances_list, idx):
        if not isinstance(instances_list, list) or idx >= len(instances_list):
            return None
        item = instances_list[idx] or {}
        arr = item.get("instances", [])
        return arr[0] if isinstance(arr, list) and len(arr) else None

    left_inst  = pick_instance(left_hand_instances,  frame_idx)
    right_inst = pick_instance(right_hand_instances, frame_idx)

    # ---- render ----
    vis_frame, naked_frame, marks_only, primitives = visualize_hands_pose(frame, left_inst, right_inst)  # you already have this
    # if vis_frame is None:
    #     vis_frame = frame  # fallback: just save the raw frame

    # ---- save PNG ----
    # os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    # # Ensure PNG extension
    # if not output_path.lower().endswith(".png"):
    #     output_path += ".png"

    # ok = cv2.imwrite(output_path, vis_frame)
    # if not ok:
    #     raise RuntimeError(f"cv2.imwrite failed for {output_path}")

    # ---- optional preview (no loop; just show once) ----
    # if preview:
    #     # Fit to a max width for display
    #     max_w = 960
    #     scale = min(1.0, max_w / vis_frame.shape[1])
    #     disp = cv2.resize(vis_frame, (int(vis_frame.shape[1]*scale), int(vis_frame.shape[0]*scale)))
    #     cv2.imshow("Hand Pose (preview)", disp)
    #     cv2.waitKey(1000)
    #     cv2.destroyAllWindows()

    # print(f"✅ Saved PNG for frame {frame_idx} to: {output_path}")
    return vis_frame, naked_frame, marks_only, primitives

def visualize_hands_pose_v2(
    frame,
    left_hand_instances,
    right_hand_instances,
    ground_truth,
    vmax=12,
    kpt_thresh=0.3,
    bone_thickness=5,
    joint_radius=6,
    outline_thick=1,
    color_fn=None,  # optional override; defaults to your color_for_value
):
    """
    Returns:
        vis_frame   : frame with markings (numpy array)
        naked_frame : original frame (no markings) (numpy array)
        vis_pdf     : PDF bytes of the vis_frame (for high-quality export)
    Notes:
        - Expects `color_for_value(v, vmin, vmax)` available if color_fn is None.
        - Coordinates are in the input frame's pixel space.
    """
    import io
    if color_fn is None:
        # use the project's mapping; assumed to be defined elsewhere
        color_fn = color_for_value

    H, W = frame.shape[:2]
    naked_frame = frame.copy()
    vis_frame = frame.copy()  # draw markings directly here for opaque overlay

    # Hand skeleton edges
    finger_connections = [
        (0, 1), (1, 2), (2, 3), (3, 4),        # thumb
        (0, 5), (5, 6), (6, 7), (7, 8),        # index
        (0, 9), (9, 10), (10, 11), (11, 12),   # middle
        (0, 13), (13, 14), (14, 15), (15, 16), # ring
        (0, 17), (17, 18), (18, 19), (19, 20)  # pinky
    ]

    # Colors from your custom mapping (green → yellow → orange → red)
    left_color_bgr  = color_fn(ground_truth.get('left_mean', 0.0),  vmin=0, vmax=vmax)
    right_color_bgr = color_fn(ground_truth.get('right_mean', 0.0), vmin=0, vmax=vmax)

    # ---- helpers ----
    def _to_xy(a):
        a = np.asarray(a)
        if a.ndim == 3 and a.shape[0] == 1:
            a = a[0]
        return a

    def _to_1d(a):
        return np.asarray(a).squeeze()

    def _safe_pts(keypoints, i, j):
        pt1 = tuple(map(int, keypoints[i]))
        pt2 = tuple(map(int, keypoints[j]))
        return pt1, pt2

    def draw_hand(instances, color_bgr):
        if instances is None:
            return
        keypoints = _to_xy(instances['keypoints'])
        scores    = _to_1d(instances['keypoint_scores'])
        n = min(len(keypoints), len(scores))
        if n == 0:
            return

        # bones
        for i, j in finger_connections:
            if i < n and j < n and scores[i] > kpt_thresh and scores[j] > kpt_thresh:
                pt1, pt2 = _safe_pts(keypoints, i, j)
                # draw directly on vis_frame
                cv2.line(vis_frame, pt1, pt2, color_bgr, bone_thickness, lineType=cv2.LINE_AA)

        # joints (outline then fill)
        for idx in range(n):
            if scores[idx] > kpt_thresh:
                x, y = map(int, keypoints[idx])

                if outline_thick > 0:
                    # outline
                    cv2.circle(vis_frame, (x, y), joint_radius + outline_thick,
                               (0, 0, 0), thickness=outline_thick, lineType=cv2.LINE_AA)

                # fill
                cv2.circle(vis_frame, (x, y), joint_radius, color_bgr, thickness=-1, lineType=cv2.LINE_AA)

    # draw both hands on vis_frame
    draw_hand(left_hand_instances,  left_color_bgr)
    draw_hand(right_hand_instances, right_color_bgr)

    # Generate PDF bytes of vis_frame
    fig = plt.figure(figsize=(W / 100.0, H / 100.0), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
    ax.set_axis_off()
    buf = io.BytesIO()
    fig.savefig(buf, format='pdf', bbox_inches='tight', pad_inches=0, dpi=100)
    plt.close(fig)
    buf.seek(0)
    vis_pdf = buf.getvalue()

    return vis_frame, naked_frame, vis_pdf

def visualize_hands_pose_v2(
    frame,
    left_hand_instances,
    right_hand_instances,
    ground_truth,
    vmax=12,
    kpt_thresh=0.3,
    bone_thickness=5,
    joint_radius=6,
    outline_thick=1,
    color_fn=None,  # optional override; defaults to your color_for_value
):
    """
    Returns:
        vis_frame   : frame with markings (numpy array)
        naked_frame : original frame (no markings) (numpy array)
        marks_only  : only the markings on black background (same size as frame)
        primitives  : dict for vector export
    Notes:
        - Expects `color_for_value(v, vmin, vmax)` available if color_fn is None.
        - Coordinates are in the input frame's pixel space.
        - Markings are drawn as vectors in PDF for sharpness on zoom.
    """
    if color_fn is None:
        color_fn = color_for_value

    H, W = frame.shape[:2]
    naked_frame = frame.copy()
    vis_frame = frame.copy()  # raster for PNG
    marks_only = np.zeros_like(frame)  # for marks_only
    primitives = {"frame_size": (W, H), "items": []}

    # Hand skeleton edges
    finger_connections = [
        (0, 1), (1, 2), (2, 3), (3, 4),        # thumb
        (0, 5), (5, 6), (6, 7), (7, 8),        # index
        (0, 9), (9, 10), (10, 11), (11, 12),   # middle
        (0, 13), (13, 14), (14, 15), (15, 16), # ring
        (0, 17), (17, 18), (18, 19), (19, 20)  # pinky
    ]

    # Colors from your custom mapping (green → yellow → orange → red)
    left_color_bgr  = color_fn(ground_truth.get('left_mean', 0.0),  vmin=0, vmax=vmax)
    right_color_bgr = color_fn(ground_truth.get('right_mean', 0.0), vmin=0, vmax=vmax)
    # Convert BGR to RGB int for primitives
    def bgr_to_rgb_triplet(bgr):
        b, g, r = bgr
        return (int(r), int(g), int(b))

    left_rgb  = bgr_to_rgb_triplet(left_color_bgr)
    right_rgb = bgr_to_rgb_triplet(right_color_bgr)

    # ---- helpers ----
    def _to_xy(a):
        a = np.asarray(a)
        if a.ndim == 3 and a.shape[0] == 1:
            a = a[0]
        return a

    def _to_1d(a):
        return np.asarray(a).squeeze()

    def _safe_pts(keypoints, i, j):
        pt1 = tuple(map(int, keypoints[i]))
        pt2 = tuple(map(int, keypoints[j]))
        return pt1, pt2

    def draw_and_record_hand(instances, color_bgr, color_rgb):
        if instances is None:
            return
        keypoints = _to_xy(instances['keypoints'])
        scores    = _to_1d(instances['keypoint_scores'])
        n = min(len(keypoints), len(scores))
        if n == 0:
            return

        # bones
        for i, j in finger_connections:
            if i < n and j < n and scores[i] > kpt_thresh and scores[j] > kpt_thresh:
                pt1, pt2 = _safe_pts(keypoints, i, j)
                # raster on vis_frame and marks_only
                cv2.line(vis_frame, pt1, pt2, color_bgr, bone_thickness, lineType=cv2.LINE_AA)
                cv2.line(marks_only, pt1, pt2, color_bgr, bone_thickness, lineType=cv2.LINE_AA)
                # vector primitive
                primitives["items"].append({
                    "type": "line",
                    "p1": (float(pt1[0]), float(pt1[1])),
                    "p2": (float(pt2[0]), float(pt2[1])),
                    "rgb": color_rgb,
                    "thick": float(bone_thickness),
                })

        # joints (outline then fill)
        for idx in range(n):
            if scores[idx] > kpt_thresh:
                x, y = map(int, keypoints[idx])

                if outline_thick > 0:
                    # raster outline
                    cv2.circle(vis_frame, (x, y), joint_radius + outline_thick,
                               (0, 0, 0), thickness=outline_thick, lineType=cv2.LINE_AA)
                    cv2.circle(marks_only, (x, y), joint_radius + outline_thick,
                               (0, 0, 0), thickness=outline_thick, lineType=cv2.LINE_AA)
                    # vector outline
                    primitives["items"].append({
                        "type": "circle",
                        "center": (float(x), float(y)),
                        "radius": float(joint_radius + outline_thick),
                        "rgb": (0, 0, 0),
                        "fill": None,
                        "thick": float(outline_thick),
                    })

                # raster fill
                cv2.circle(vis_frame, (x, y), joint_radius, color_bgr, thickness=-1, lineType=cv2.LINE_AA)
                cv2.circle(marks_only, (x, y), joint_radius, color_bgr, thickness=-1, lineType=cv2.LINE_AA)
                # vector fill
                primitives["items"].append({
                    "type": "circle",
                    "center": (float(x), float(y)),
                    "radius": float(joint_radius),
                    "rgb": color_rgb,
                    "fill": color_rgb,
                    "thick": 0.0,
                })

    # draw both hands
    draw_and_record_hand(left_hand_instances,  left_color_bgr,  left_rgb)
    draw_and_record_hand(right_hand_instances, right_color_bgr, right_rgb)

    return vis_frame, naked_frame, primitives

def process_png_from_json_v2(video_path, hand_output_json_path, ground_truth, vmax=12, frame_idx=0):

    # ---- load detections ----
    with open(hand_output_json_path, "r") as f:
        data = json.load(f)

    left_hand_instances  = data.get("left_hand_instances", [])
    right_hand_instances = data.get("right_hand_instances", [])

    # ---- open video & sanity checks ----
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total_frames <= 0:
        raise ValueError(f"Video has no frames: {video_path}")

    if frame_idx < 0 or frame_idx >= total_frames:
        raise IndexError(f"frame_idx {frame_idx} out of range [0, {total_frames-1}]")

    # ---- seek & read frame ----
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        raise RuntimeError(f"Could not read frame {frame_idx} from {video_path}")

    # ---- pick instances for this frame (robust to missing/empty) ----
    def pick_instance(instances_list, idx):
        if not isinstance(instances_list, list) or idx >= len(instances_list):
            return None
        item = instances_list[idx] or {}
        arr = item.get("instances", [])
        return arr[0] if isinstance(arr, list) and len(arr) else None

    left_inst  = pick_instance(left_hand_instances,  frame_idx)
    right_inst = pick_instance(right_hand_instances, frame_idx)

    # ---- render ----
    vis_frame, naked_frame, vis_pdf = visualize_hands_pose_v2(frame, left_inst, right_inst, ground_truth, vmax)

    return vis_frame, naked_frame, vis_pdf

def compute_vmax_from_means(results, methods, floor=12.0):
    """
    Use only left_mean/right_mean from results[method], ignore dists.
    Returns max(floor, max_means).
    """
    max_mean = float('-inf')
    for m in methods:
        r = results.get(m, {})
        for key in ("left_mean", "right_mean"):
            v = r.get(key, None)
            if v is None:
                continue
            try:
                v = float(v)
            except (TypeError, ValueError):
                continue
            if not math.isnan(v):
                if v > max_mean:
                    max_mean = v
    if max_mean == float('-inf'):
        return float(floor)  # no means found -> default to floor
    return max(floor, max_mean)

## Make png of specific frame (normal)

In [ ]:
pose_output_dir = 'predictions_2d/'
vis_output_dir = 'vis_dir/'
method = 'dwpose'
data_collection = 'cha/cha3'
video_names = ['gopro5','gopro6','gopro7','gopro8','gopro9','gopro10','gopro11','gopro12']
video_names = ['gopro7']
frame = 144

for video_name in video_names:
    video_path = f'inputs/{data_collection}/{video_name}_synced_cut.MP4'
    hand_output_json_path = os.path.join(pose_output_dir, f"{method}", data_collection, f"{video_name}_synced_cut_wholebody.json")
    output_frames_path = os.path.join(vis_output_dir, f"{method}", data_collection, f"{video_name}_frame_{frame}.png")
    print(hand_output_json_path, output_frames_path)


    process_png_from_json(video_path, hand_output_json_path, output_path=output_frames_path, frame_idx=frame, preview=True)

### Plot 4 different method frames side by side

In [5]:
import os, cv2, numpy as np

pose_output_dir = 'predictions_2d/'
vis_output_dir  = 'vis_dir/'
methods = ['rtmpose','dwpose','refined_poses','rtmpose_hands']  # exactly 4
data_collection = 'cha/cha3'
video_names = ['gopro7']
frame = 144

# display names
display_names = {
    'rtmpose': 'RTMPose',
    'dwpose': 'DWPose',
    'refined_poses': 'Proposed',
    'rtmpose_hands': 'RTMPose -> Hands',
}

# canvas config
canvas_w, canvas_h = 3840, 2160
tile_w, tile_h = canvas_w // 2, canvas_h // 2
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 2.2
thick = 5
text_color = (0, 255, 255)  # BGR yellow
pad = 28

# TL, BL, TR, BR
positions = [(0,0), (1,0), (0,1), (1,1)]

def add_top_right_label(img, text):
    text = str(text)
    (tw, th), _ = cv2.getTextSize(text, font, font_scale, thick)
    x = img.shape[1] - tw - pad
    y = pad + th
    cv2.putText(img, text, (x, y), font, font_scale, text_color, thick, cv2.LINE_AA)
    return img

def resize_tile(img):
    return cv2.resize(img, (tile_w, tile_h), interpolation=cv2.INTER_AREA)

# --- collect per-video results ---
results_by_video = {vn: {"vis": [], "naked": [], "marks": [], "prims": []} for vn in video_names}

for video_name in video_names:
    for method in methods:
        video_path = f'inputs/{data_collection}/{video_name}_synced_cut.MP4'
        hand_output_json_path = os.path.join(
            pose_output_dir, f"{method}", data_collection, f"{video_name}_synced_cut_wholebody.json"
        )
        output_frames_path = os.path.join(
            vis_output_dir, f"{method}", data_collection, f"{video_name}_frame_{frame}.png"
        )
        vis_frame, naked_frame, marks_only, primitives = process_png_from_json(
            video_path, hand_output_json_path, output_path=output_frames_path, frame_idx=frame, preview=False
        )
        results_by_video[video_name]["vis"].append(vis_frame)
        results_by_video[video_name]["naked"].append(naked_frame)
        results_by_video[video_name]["marks"].append(marks_only)
        results_by_video[video_name]["prims"].append(primitives)

# --- vector PDF writer (text + joints/lines only; no legend) ---
def write_vector_pdf_without_legend(out_pdf_path, canvas_w, canvas_h, tile_w, tile_h,
                                    positions, methods, display_names, prims_frames):
    try:
        from reportlab.pdfgen import canvas as rl_canvas
        from reportlab.lib.colors import Color
    except ImportError:
        print("ℹ️ ReportLab not installed. Install with: pip install reportlab")
        return

    c = rl_canvas.Canvas(out_pdf_path, pagesize=(canvas_w, canvas_h))

    # helpers
    def to_color(rgb):
        r, g, b = rgb
        return Color(r/255.0, g/255.0, b/255.0)

    def draw_line(p1, p2, color, thick):
        c.setStrokeColor(to_color(color))
        c.setLineWidth(float(thick))
        c.line(p1[0], p1[1], p2[0], p2[1])

    def draw_circle(center, radius, stroke_color, stroke_w, fill_color=None):
        if fill_color is not None:
            c.setFillColor(to_color(fill_color))
        else:
            c.setFillColor(Color(0,0,0,alpha=0))
        c.setStrokeColor(to_color(stroke_color))
        c.setLineWidth(float(stroke_w))
        c.circle(center[0], center[1], float(radius), stroke=1, fill=1 if fill_color is not None else 0)

    # label sizing (relative to tile height)
    label_font = "Helvetica-Bold"
    label_size = max(24, int(tile_h * 0.06))
    c.setFont(label_font, label_size)
    pad = 28
    ascent_factor = 0.80  # ~Helvetica ascent ratio

    for idx, (r, col) in enumerate(positions):
        # --- label: top-right inside the tile, baseline adjusted for ascent ---
        label = str(display_names.get(methods[idx], methods[idx]))
        tw = c.stringWidth(label, label_font, label_size)
        # tile top edge in page coords
        tile_top = canvas_h - r * tile_h
        # baseline Y = top - pad - ascent (so text stays inside)
        ascent = label_size * ascent_factor
        tx = (col + 1) * tile_w - pad
        ty = tile_top - pad - ascent
        c.setFillColorRGB(1, 1, 0)  # yellow (match PNG)
        c.drawString(tx - tw, ty, label)

        # --- primitives (vector) ---
        prim = prims_frames[idx] if idx < len(prims_frames) else None
        if not prim:
            continue

        src_w, src_h = prim["frame_size"]
        sx = tile_w / float(src_w)
        sy = tile_h / float(src_h)

        tile_left = col * tile_w
        def xf(x): return tile_left + x * sx
        def yf(y): return tile_top - y * sy  # invert y

        for item in prim["items"]:
            typ = item.get("type")
            if typ == "line":
                x1, y1 = item["p1"]; x2, y2 = item["p2"]
                rgb = tuple(int(v) for v in item["rgb"])
                thick = float(item.get("thick", 2.0)) * ((sx + sy) / 2.0)
                draw_line((xf(x1), yf(y1)), (xf(x2), yf(y2)), rgb, thick)
            elif typ == "circle":
                x, y = item["center"]
                rad  = float(item["radius"]) * ((sx + sy) / 2.0)
                rgb  = tuple(int(v) for v in item["rgb"])
                thick = float(item.get("thick", 0.0)) * ((sx + sy) / 2.0)
                fill = item.get("fill", None)
                fill_rgb = tuple(int(v) for v in fill) if fill is not None else None
                draw_circle((xf(x), yf(y)), rad, rgb, thick, fill_rgb)

    c.showPage()
    c.save()


# --- build & save per-video collages and vector PDF ---
for video_name in video_names:
    vis_list   = results_by_video[video_name]["vis"]
    naked_list = results_by_video[video_name]["naked"]
    prims_list = results_by_video[video_name]["prims"]

    if len(vis_list) != 4:
        raise ValueError("Please provide exactly 4 methods per video.")

    # 1) Final collage (labels + joints)
    tiles_labeled = [add_top_right_label(resize_tile(vis_list[i]), display_names.get(m, m))
                     for i, m in enumerate(methods)]
    final_canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    for idx, (r, c) in enumerate(positions):
        y0, y1 = r * tile_h, (r + 1) * tile_h
        x0, x1 = c * tile_w, (c + 1) * tile_w
        final_canvas[y0:y1, x0:x1] = tiles_labeled[idx]

    # 2) Naked collage (no labels, no joints)
    tiles_naked = [resize_tile(naked_list[i]) for i in range(4)]
    naked_canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    for idx, (r, c) in enumerate(positions):
        y0, y1 = r * tile_h, (r + 1) * tile_h
        x0, x1 = c * tile_w, (c + 1) * tile_w
        naked_canvas[y0:y1, x0:x1] = tiles_naked[idx]

    # 3) Vector PDF with text + joints/lines only (no legend)
    out_dir = os.path.join(vis_output_dir, "collages", data_collection, f"{video_name}_frame_{frame}_nocomp")
    os.makedirs(out_dir, exist_ok=True)
    final_png_path = os.path.join(out_dir, "final_colored.png")
    naked_png_path = os.path.join(out_dir, "naked.png")
    pdf_path       = os.path.join(out_dir, "markings_vector.pdf")

    # save PNGs
    cv2.imwrite(final_png_path, final_canvas)
    cv2.imwrite(naked_png_path, naked_canvas)

    # save vector PDF (text + primitives)
    write_vector_pdf_without_legend(
        out_pdf_path=pdf_path,
        canvas_w=canvas_w, canvas_h=canvas_h,
        tile_w=tile_w, tile_h=tile_h,
        positions=positions,
        methods=methods,
        display_names=display_names,
        prims_frames=prims_list
    )

    print(f"✅ Saved: {final_png_path}")
    print(f"✅ Saved: {naked_png_path}")
    print(f"🧾 Saved: {pdf_path}")


NameError: name 'pt1' is not defined

## Make full Video

In [ ]:
pose_output_dir = 'predictions_2d/'
vis_output_dir = 'vis_dir/'
method = 'refined_poses'
data_collection = 'cha/cha1'
video_names = ['gopro5','gopro6','gopro7','gopro8','gopro9','gopro10','gopro11','gopro12']
video_names = ['gopro6','gopro7','gopro9','gopro12']
for video_name in video_names:
    video_path = f'inputs/{data_collection}/{video_name}_synced_cut.MP4'
    hand_output_json_path = os.path.join(pose_output_dir, f"{method}", data_collection, f"{video_name}_synced_cut_wholebody.json")
    output_frames_path = os.path.join(vis_output_dir, f"{method}", data_collection, f"{video_name}_new.mp4")
    print(hand_output_json_path, output_frames_path)

    if os.path.exists(output_frames_path):
        print(f"Skipping {video_name}, output already exists at {output_frames_path}")
        continue

    process_video_from_json(video_path, hand_output_json_path, output_path=output_frames_path, stride=1, preview=True)

# Plot 4 different methods colored by pixel error

### Compare 2D annotations to ground truth

In [1]:
import json
from pathlib import Path
from helpers.json_handling import extract_specific_frame, per_joint_distances
import numpy as np
import os
camera = "gopro7"
frame = 144
input = 'cha/cha3'
methods = ['dwpose','rtmpose','rtmpose_hands','refined_poses']

dataset_file = Path(f"../repo-labeling/output_3d/single_frames/{input[4:]}_{camera}_{frame}/hand_poses_2d_fixed.npz")
ground_truth = extract_specific_frame(dataset_file) # extracts only the data for the specific frame that was labeled
print(ground_truth["camera"], ground_truth["frame_index"])

left_gt = ground_truth["hands"][0] #left hand keypoints
right_gt = ground_truth["hands"][1] #right hand keypoints
# print("left gt", left_gt['keypoint_scores'])
# print("right gt", right_gt['keypoint_scores'])
results = {}  

for method in methods:
    # Read (load) a JSON file into a Python object (dict/list)
    path = Path(f"predictions_2d/{method}/{input}/{camera}_synced_cut_wholebody.json")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)   # -> dict/list/etc.

    left = data["left_hand_instances"][frame]['instances'][0]
    right = data["right_hand_instances"][frame]['instances'][0]

    right_score = [1 if x > 0.3 else 0 for x in right['keypoint_scores']]
    left_score = [1 if x > 0.3 else 0 for x in left['keypoint_scores']]

    left_dists  = per_joint_distances(left,  left_gt,  threshold=0.3)
    right_dists = per_joint_distances(right, right_gt, threshold=0.3)

    # print("Left per-joint distances:", left_dists)
    # print("Right per-joint distances:", right_dists)
    print(np.nanmean(left_dists))
    print(np.nanmean(right_dists))

    results[method] = {
        "left_dists":  left_dists.tolist(),
        "right_dists": right_dists.tolist(),
        "left_mean":   float(np.nanmean(left_dists))  if left_dists.size  else float("nan"),
        "right_mean":  float(np.nanmean(right_dists)) if right_dists.size else float("nan"),
    }
    

FileNotFoundError: [Errno 2] No such file or directory: '..\\repo-labeling\\output_3d\\single_frames\\cha3_gopro7_144\\hand_poses_2d_fixed.npz'

In [116]:
import os, cv2, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from matplotlib.collections import LineCollection
import io

pose_output_dir = 'predictions_2d/'
vis_output_dir  = 'vis_dir/'
methods = ['rtmpose','dwpose','refined_poses','rtmpose_hands']  # exactly 4

# --- display names you can change ---
display_names = {
    'rtmpose': 'RTMPose',
    'dwpose': 'DWPose',
    'refined_poses': 'Proposed',
    'rtmpose_hands': 'RTMPose -> Hands',
}

# ---- render each method's annotated frame (must return: vis, naked, primitives) ----
vis_frames   = []
naked_frames = []
prims_frames = []

vmax = compute_vmax_from_means(results, methods, floor=12.0)

for method in methods:
    video_path = f'inputs/{input}/{camera}_synced_cut.MP4'
    hand_output_json_path = os.path.join(pose_output_dir, f"{method}", input, f"{camera}_synced_cut_wholebody.json")
    vis_frame, naked_frame, primitives = process_png_from_json_v2(
        video_path, hand_output_json_path, results[method], vmax, frame_idx=frame
    )
    vis_frames.append(vis_frame)
    naked_frames.append(naked_frame)
    prims_frames.append(primitives)


# --- collage config ---
canvas_w, canvas_h = 3840, 2160
tile_w, tile_h = canvas_w // 2, canvas_h // 2   # 1920x1080
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 2.2
thick = 5
text_color = (0, 255, 255)  # yellow (BGR)
pad = 28

# Positions in requested order: 1st->TL, 2nd->BL, 3rd->TR, 4th->BR
positions = [(0,0), (1,0), (0,1), (1,1)]

def add_top_right_label(img, text):
    text = str(text)
    (tw, th), _ = cv2.getTextSize(text, font, font_scale, thick)
    x = img.shape[1] - tw - pad
    y = pad + th
    cv2.putText(img, text, (x, y), font, font_scale, text_color, thick, cv2.LINE_AA)
    return img

def resize_tile(img):
    return cv2.resize(img, (tile_w, tile_h), interpolation=cv2.INTER_AREA)

def load_and_prepare(frame, method_key):
    if frame is None:
        raise RuntimeError(f"Empty image for {method_key}")
    img = resize_tile(frame)
    label = display_names.get(method_key, method_key)
    return add_top_right_label(img, label)

# ---------- legend (white) with centered title ON the bar, dynamic ticks based on vmax ----------
def make_legend(width, vmin=0.0, vmax=12.0, font=cv2.FONT_HERSHEY_SIMPLEX):
    pad          = 28
    legend_pad   = 30
    legend_bar_h = 70
    legend_h     = legend_bar_h + 2*legend_pad + 60
    small_scale  = 1.2
    small_thick  = 3

    legend = np.full((legend_h, width, 3), 255, dtype=np.uint8)

    bar_x0 = pad
    bar_x1 = width - pad
    bar_w  = max(1, bar_x1 - bar_x0)
    bar_y0 = legend_pad
    bar_y1 = bar_y0 + legend_bar_h

    # gradient using your color_for_value(...)
    bar = np.zeros((legend_bar_h, bar_w, 3), dtype=np.uint8)
    for x in range(bar_w):
        t = x / (bar_w - 1) if bar_w > 1 else 0.0
        v = vmin + t * (vmax - vmin)
        bar[:, x] = color_for_value(v, vmin=vmin, vmax=vmax)
    legend[bar_y0:bar_y1, bar_x0:bar_x1] = bar

    cv2.rectangle(legend, (bar_x0, bar_y0), (bar_x1, bar_y1), (180, 180, 180), 2, cv2.LINE_AA)

    # dynamic ticks based on vmax
    def _fmt_tick(val):
        return f"{int(round(val))}" if abs(val - round(val)) < 1e-6 else f"{val:.1f}"

    vmax = max(vmin + 1e-6, float(vmax))
    n_ticks = 5
    tick_vals = np.linspace(vmin, vmax, n_ticks)

    for i, v in enumerate(tick_vals):
        x = int(bar_x0 + ((v - vmin) / (vmax - vmin)) * (bar_w - 1))
        cv2.line(legend, (x, bar_y1), (x, bar_y1 + 12), (0, 0, 0), 2, cv2.LINE_AA)
        label_core = _fmt_tick(v)
        label = f"{label_core} px" if i in (0, n_ticks - 1) else label_core
        (tw, th), _ = cv2.getTextSize(label, font, small_scale, small_thick)
        tx = int(np.clip(x - tw // 2, pad, width - pad - tw))
        ty = bar_y1 + 12 + th + 6
        cv2.putText(legend, label, (tx, ty), font, small_scale, (0, 0, 0), small_thick, cv2.LINE_AA)

    # centered title on the bar (black)
    title = "Mean distance to ground truth [px]"
    (tw, th), _ = cv2.getTextSize(title, font, 1.0, 2)
    tx = bar_x0 + (bar_w - tw) // 2
    ty = bar_y0 + (legend_bar_h + th) // 2
    cv2.putText(legend, title, (tx, ty), font, 1.0, (0, 0, 0), 2, cv2.LINE_AA)

    return legend

# ---------- place legend ABOVE canvas, aligned top-right ----------
def place_legend_above_top_right(base_canvas, legend_img, margin=24, header_pad=16, draw_separator=True):
    H, W = base_canvas.shape[:2]
    lh, lw = legend_img.shape[:2]

    # Header height = legend height + vertical padding
    header_h = lh + 2 * header_pad

    # New canvas (white header + original)
    final_canvas = np.full((header_h + H, W, 3), 255, dtype=np.uint8)
    final_canvas[header_h:header_h + H, :W] = base_canvas

    # If legend too wide, shrink to fit with margins
    x1 = W - margin
    x0 = x1 - lw
    if x0 < 0:
        scale = (W - 2*margin) / lw
        legend_img = cv2.resize(legend_img, (int(lw*scale), int(lh*scale)), interpolation=cv2.INTER_AREA)
        lh, lw = legend_img.shape[:2]
        x1 = W - margin
        x0 = x1 - lw

    y0 = header_pad
    y1 = y0 + lh
    final_canvas[y0:y1, x0:x1] = legend_img

    if draw_separator:
        cv2.line(final_canvas, (0, header_h - 1), (W, header_h - 1), (180, 180, 180), 2, cv2.LINE_AA)

    return final_canvas

# ---- build the 2x2 collages ----
if len(methods) != 4:
    raise ValueError("Please provide exactly 4 methods.")

# 1) Final collage (labels ON tiles)
tiles_labeled = [load_and_prepare(vis_frames[i], m) for i, m in enumerate(methods)]
final_base = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
for idx, (r, c) in enumerate(positions):
    y0, y1 = r * tile_h, (r + 1) * tile_h
    x0, x1 = c * tile_w, (c + 1) * tile_w
    final_base[y0:y1, x0:x1] = tiles_labeled[idx]

legend_width = 1100
legend_img  = make_legend(legend_width, vmin=0.0, vmax=vmax)
final_canvas = place_legend_above_top_right(final_base, legend_img, margin=24, header_pad=18)

# 2) Naked collage (no labels, no legend)
tiles_naked = [resize_tile(naked_frames[i]) for i in range(4)]
naked_canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
for idx, (r, c) in enumerate(positions):
    y0, y1 = r * tile_h, (r + 1) * tile_h
    x0, x1 = c * tile_w, (c + 1) * tile_w
    naked_canvas[y0:y1, x0:x1] = tiles_naked[idx]


# ---- output folder: vis_dir/collages/<input>/<camera>_frame_<frame>/ ----
base_dir = os.path.join(vis_output_dir, "collages", input)
out_dir = os.path.join(base_dir, f"{camera}_frame_{frame}")
os.makedirs(out_dir, exist_ok=True)

# Save PNGs
final_png_path = os.path.join(out_dir, "final_colored_new.png")
naked_png_path = os.path.join(out_dir, "naked.png")
cv2.imwrite(final_png_path, final_canvas)
cv2.imwrite(naked_png_path, naked_canvas)

print(f"✅ Saved final PNG: {final_png_path}")
print(f"✅ Saved naked PNG: {naked_png_path}")

# ====================================================================================
#                           VECTOR PDF COLLAGE (high-quality: vector markings, labels, legend)
# ====================================================================================

def draw_top_right_label(ax, label_text, img_w, pad_px=28, fontsize=28, textcolor='yellow'):
    """Vector label mimicking your OpenCV label in the PNG."""
    x = img_w - pad_px
    y = pad_px + fontsize * 1.2
    ax.text(
        x, y, label_text,
        ha='right', va='top',
        fontsize=fontsize, color=textcolor,
        fontweight='bold',
        bbox=dict(facecolor='none', edgecolor='none', pad=0.0)
    )

def draw_primitives_on_axes(ax, prims, default_color=(0,1,0), default_lw=2.0, alpha=1.0):
    """
    Draw vector primitives contained in `prims` onto Matplotlib axes `ax`.
    Assumes prims = {"frame_size": (W,H), "items": [...] } with items having "type", "p1", "p2", "rgb":(r,g,b), etc.
    """
    if not prims or "items" not in prims:
        return
    for p in prims["items"]:
        ptype = p.get("type", "").lower()
        rgb = p.get("rgb", default_color)
        color_norm = tuple(c / 255.0 for c in rgb)
        if ptype == "line":
            (x1,y1) = p["p1"]; (x2,y2) = p["p2"]
            ax.plot([x1,x2], [y1,y2],
                    color=color_norm,
                    lw=p.get("thick", default_lw),
                    alpha=alpha, solid_capstyle='round')
        elif ptype == "circle":
            (cx,cy) = p["center"]; r = float(p["radius"])
            facecolor = tuple(c / 255.0 for c in p.get("fill", (0,0,0))) if p.get("fill") else 'none'
            circ = Circle((cx,cy), r,
                          facecolor=facecolor,
                          edgecolor=color_norm,
                          linewidth=p.get("thick", default_lw),
                          alpha=alpha)
            ax.add_patch(circ)

def draw_vector_legend(ax, width, vmin=0.0, vmax=12.0, margin_right=24, header_pad=18, header_h=None):
    """
    Draw vector legend on the given axes (header axes), matching raster appearance.
    """
    legend_width = 1100
    pad = 28
    legend_pad = 30
    legend_bar_h = 70
    legend_h = legend_bar_h + 2*legend_pad + 60

    if header_h is None:
        header_h = legend_h + 2*header_pad
    # Position: top-right, vertically centered in header (y from bottom)
    leg_x1 = width - margin_right
    leg_x0 = leg_x1 - legend_width
    # Vertical position from bottom: header_pad to header_h - header_pad
    leg_y0 = header_pad
    leg_y1 = leg_y0 + legend_h

    # White background for legend card
    ax.add_patch(Rectangle((leg_x0, leg_y0), legend_width, legend_h, facecolor='white', edgecolor='none'))

    # Legend card border
    ax.add_patch(Rectangle((leg_x0, leg_y0), legend_width, legend_h, facecolor='none', edgecolor=(180/255,180/255,180/255), lw=2))

    # Gradient bar
    bar_x0 = leg_x0 + pad
    bar_x1 = leg_x0 + legend_width - pad
    bar_w = bar_x1 - bar_x0
    bar_y0 = leg_y0 + legend_pad
    bar_y1 = bar_y0 + legend_bar_h

    # Draw gradient as thin rectangles
    for x in range(int(bar_w)):
        t = x / (bar_w - 1) if bar_w > 1 else 0.0
        v = vmin + t * (vmax - vmin)
        color_bgr = color_for_value(v, vmin=vmin, vmax=vmax)
        color_rgb_norm = (color_bgr[2]/255.0, color_bgr[1]/255.0, color_bgr[0]/255.0)
        ax.add_patch(Rectangle((bar_x0 + x, bar_y0), 1, legend_bar_h, facecolor=color_rgb_norm, edgecolor='none'))

    # Bar outline
    ax.add_patch(Rectangle((bar_x0, bar_y0), bar_w, legend_bar_h, facecolor='none', edgecolor=(180/255,180/255,180/255), lw=2))

    # Ticks and labels
    def _fmt_tick(val):
        return f"{int(round(val))}" if abs(val - round(val)) < 1e-6 else f"{val:.1f}"

    n_ticks = 5
    tick_vals = np.linspace(vmin, vmax, n_ticks)

    for i, v in enumerate(tick_vals):
        x = bar_x0 + ((v - vmin) / (vmax - vmin)) * bar_w
        # Tick line (from bar_y1 up to bar_y1 + 12)
        ax.plot([x, x], [bar_y1, bar_y1 + 12], color='black', lw=2)
        # Label
        label_core = _fmt_tick(v)
        label = f"{label_core} px" if i in (0, n_ticks - 1) else label_core
        ax.text(x, bar_y1 + 44, label, ha='center', va='bottom', fontsize=20, color='black')

    # Centered title on the bar
    title = "Mean distance to ground truth [px]"
    ax.text((bar_x0 + bar_x1)/2, (bar_y0 + bar_y1)/2, title, ha='center', va='center', fontsize=22, fontweight='bold', color='black')

    # Separator line below header (at header_h)
    ax.plot([0, width], [header_h, header_h], color=(180/255,180/255,180/255), lw=2)

def save_vector_collage_pdf(out_pdf_path, naked_frames, prims_frames, vmax, methods, positions, display_names,
                            canvas_w=3840, canvas_h=2160, tile_w=1920, tile_h=1080, inkscape_px=True):
    """
    inkscape_px=True -> make the PDF page 1:1 with 'px' at 96 DPI so Inkscape shows ~3840x... directly.
    Vector overlays remain vector; rasters are placed 1 px = 1 unit at 96 DPI.
    """

    # --- header geometry (same as before) ---
    legend_width = 1100
    legend_pad   = 30
    legend_bar_h = 70
    legend_h     = legend_bar_h + 2*legend_pad + 60
    header_pad   = 18
    header_h     = legend_h + 2*header_pad

    total_w_px = canvas_w
    total_h_px = canvas_h + header_h

    if inkscape_px:
        # Make page size in inches so that px == Inkscape px (96 DPI)
        dpi = 96
        fig_w_in = total_w_px / dpi
        fig_h_in = total_h_px / dpi
    else:
        # "Print" setup (300 PPI); Inkscape will show smaller px numbers
        dpi = 300
        fig_w_in = total_w_px / dpi
        fig_h_in = total_h_px / dpi

    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    import io, cv2, numpy as np

    fig = plt.figure(figsize=(fig_w_in, fig_h_in), dpi=dpi)
    fig.patch.set_alpha(0)

    # Header axes
    ax_header = fig.add_axes([0, canvas_h / total_h_px, 1, header_h / total_h_px])
    ax_header.set_axis_off()
    ax_header.set_xlim(0, total_w_px)
    ax_header.set_ylim(header_h, 0)
    ax_header.add_patch(Rectangle((0, 0), total_w_px, header_h, facecolor='white', edgecolor='none'))
    draw_vector_legend(ax_header, total_w_px, vmin=0.0, vmax=vmax, margin_right=24, header_pad=header_pad, header_h=header_h)

    # Main canvas axes
    ax_main = fig.add_axes([0, 0, 1, canvas_h / total_h_px])
    ax_main.set_axis_off()
    ax_main.set_xlim(0, total_w_px)
    ax_main.set_ylim(canvas_h, 0)

    # Place tiles (same logic as your code)
    for idx, (r, c) in enumerate(positions):
        naked_img = naked_frames[idx]
        prims     = prims_frames[idx]
        label     = display_names.get(methods[idx], methods[idx])

        # raster background (tile size is 1920x1080 on a 3840x2160 canvas)
        resized_naked = cv2.resize(naked_img, (tile_w, tile_h), interpolation=cv2.INTER_AREA)

        tile_x0 = c * tile_w
        tile_y0 = r * tile_h
        y_top   = canvas_h - tile_y0
        y_bot   = y_top - tile_h

        ax_main.imshow(cv2.cvtColor(resized_naked, cv2.COLOR_BGR2RGB),
                       extent=[tile_x0, tile_x0 + tile_w, y_top, y_bot],
                       origin='upper', interpolation='bilinear')

        # scale and draw vector primitives on an overlay axes
        src_w, src_h = prims["frame_size"]
        sx = tile_w / src_w
        sy = tile_h / src_h
        scaled_prims = {"frame_size": (tile_w, tile_h), "items": []}
        for item in prims["items"]:
            pp = item.copy()
            if pp["type"] == "line":
                pp["p1"] = (pp["p1"][0] * sx, pp["p1"][1] * sy)
                pp["p2"] = (pp["p2"][0] * sx, pp["p2"][1] * sy)
                pp["thick"] *= (sx + sy) / 2.0
            elif pp["type"] == "circle":
                pp["center"] = (pp["center"][0] * sx, pp["center"][1] * sy)
                pp["radius"] *= (sx + sy) / 2.0
                if pp.get("fill"):
                    pp["fill"] = pp["fill"]
                pp["thick"] *= (sx + sy) / 2.0
            scaled_prims["items"].append(pp)

        bottom_frac = (canvas_h - (tile_y0 + tile_h)) / total_h_px
        ax_tile = fig.add_axes([tile_x0 / total_w_px, bottom_frac, tile_w / total_w_px, tile_h / total_h_px])
        ax_tile.set_axis_off()
        ax_tile.set_xlim(0, tile_w)
        ax_tile.set_ylim(tile_h, 0)

        draw_primitives_on_axes(ax_tile, scaled_prims, default_color=(0,1,0), default_lw=2.5, alpha=1.0)
        draw_top_right_label(ax_tile, label, img_w=tile_w, pad_px=28, fontsize=50, textcolor='yellow')

    # Save PDF — IMPORTANT: no tight bbox (keeps exact page size)
    fig.savefig(out_pdf_path, format='pdf', bbox_inches=None, pad_inches=0, dpi=dpi)
    plt.close(fig)
    print(f"✅ Saved vector collage PDF: {out_pdf_path}")

# --- Generate and save vector collage PDF ---
final_pdf_path = os.path.join(out_dir, "final_colored_vector.pdf")
save_vector_collage_pdf(final_pdf_path, naked_frames, prims_frames, vmax, methods, positions, display_names)

✅ Saved final PNG: vis_dir/collages\cha/cha4\gopro11_frame_108\final_colored_new.png
✅ Saved naked PNG: vis_dir/collages\cha/cha4\gopro11_frame_108\naked.png
✅ Saved vector collage PDF: vis_dir/collages\cha/cha4\gopro11_frame_108\final_colored_vector.pdf
